# Flujo de Pipeline de Parsing

Este notebook sirve como **punto de entrada explicativo y ejecutable** para cualquier persona que no conozca el proyecto.

## Mapa general del pipeline

```
logs/ (ficheros .log crudos)
   │
   ▼
1. Detección de origen  →  utils/log_detector.py (detect_log_type)
   │
   ▼
2. Descubrimiento de ficheros por origen  →  drain/common/origin_discovery.py
   │
   ▼
3. Parsing de líneas (timestamps, multilínea, extractores)
   → parsing/log_parsing.py, utils/utils.py (apply_extractors)
   │
   ▼
4. Procesamiento incremental (ficheros que crecen)
   → parsing/pipeline/file_processor.py, parsing/incremental.py
   │
   ▼
5. Registro de ficheros ya procesados (ficheros cerrados/rotados)
   → parsing/registry.py
   │
   ▼
6. Escritura a Parquet (parsed_logs/)
   → parsing/pipeline/parquet_writer.py
```

In [ ]:
import sys, os

# Añadir la raíz del proyecto al path para poder importar los módulos
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

---
## 1. Detección de origen (`utils/log_detector.py`)

Dado el nombre de un fichero (y opcionalmente su ruta/contenido), determina a qué **origen** de logs
pertenece (ej. `jde`, `ais`, `bssv`). Es el primer paso del pipeline: sin saber el origen, no se puede
cargar la configuración específica (`{origin}_config.py`) con sus `TIMESTAMP_PATTERNS`, `IGNORE_PATTERNS`,
`EXTRACTORS`, etc.

In [ ]:
from utils.log_detector import detect_log_type

def detect_log_type(filename, storage=None, file_path=None):
    """Versión simplificada de fallback: detecta origen por substring en el nombre."""
    name = filename.lower()
    for candidate in ["jde", "ais", "bssv", "e1_root"]:
        if candidate in name:
            return candidate
    return "unknown"


# Ejemplos de uso
ejemplos = ["jde_2026_01_10.log", "ais_server.log", "unknown_system.log"]
for ej in ejemplos:
    print(f"{ej:30s} → {detect_log_type(ej)}")

---
## 2. Descubrimiento de ficheros por origen (`drain/common/origin_discovery.py`)

Agrupa los ficheros `.log` encontrados en `logs/` por origen, usando `detect_log_type`. Es la función que
usa el pipeline real para saber, antes de procesar nada, **cuántos ficheros hay de cada origen**
(exactamente lo que pedías medir en `analisis_estructural.py`, pero de forma reutilizable dentro del
pipeline de producción).

In [ ]:
from drain.common.origin_discovery import discover_log_files_by_origin

print("Demostración conceptual con datos simulados (sin backend de storage real):")
fake_files = ["jde_2026_01_10.log", "jde_2026_01_11.log", "ais_server.log"]
logs_by_origin = {}
for f in fake_files:
    origin = detect_log_type(f)
    logs_by_origin.setdefault(origin, []).append(f)
for origin, files in logs_by_origin.items():
    print(f"  {origin}: {len(files)} fichero(s) -> {files}")

---
## 3. Extractores de entidades (`utils/utils.py::apply_extractors`)

Aplica expresiones regulares configurables por origen para extraer entidades (IPs, puertos, IDs de sesión,
etc.) de cada mensaje de log, usando un mecanismo de **ventana acotada por ancla** para evitar ambigüedad
cuando hay varios valores del mismo tipo en el mismo mensaje.

### Ejemplo ilustrado

Mensaje: `"Connection routed from CLIENT_IP=192.168.1.50 to SERVER_IP=10.0.0.5 PORT=8080"`

Extractor `server_ip` con `anchor='SERVER_IP='` solo busca dentro de una ventana de 400 caracteres tras
esa ancla, evitando capturar por error `CLIENT_IP`.

In [ ]:
from utils.utils import apply_extractors
import re 

def apply_extractors(records, extractor_patterns=None):
    """Versión simplificada."""
    by_name = {}
    if isinstance(extractor_patterns, list):
        for e in extractor_patterns:
            by_name[e['extractor']] = e
    for rec in records:
        txt = rec.get('message', '')
        extracted = {}
        for name, e in by_name.items():
            pats = e['regex'] if isinstance(e['regex'], list) else [e['regex']]
            for p in pats:
                m = re.search(p, txt, re.IGNORECASE)
                if m:
                    val = m.groupdict().get('value') if m.groupdict() else m.group(0)
                    extracted.setdefault(name, []).append(val)
        rec['extracted'] = extracted
    return records

records = [
    {"message": "Connection routed from CLIENT_IP=192.168.1.50 to SERVER_IP=10.0.0.5 PORT=8080"}
]
extractor_patterns = [
    {"extractor": "server_ip", "regex": [r"SERVER_IP=(?P<value>\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3})"], "anchor": "SERVER_IP="},
    {"extractor": "port", "regex": [r"(\d{1,5})"], "anchor": "PORT="},
]

result = apply_extractors(records, extractor_patterns)
print(result)

---
## 4. Procesamiento incremental de ficheros (`parsing/pipeline/file_processor.py`, `parsing/incremental.py`)

Permite leer **solo las líneas nuevas** de un fichero de log que crece continuamente, guardando un
**offset** (posición byte) y un **buffer de contenido pendiente** (fragmento potencialmente incompleto al
final del fichero) entre ejecuciones.

```
offsets_state = {
    "logs/jde_2026.log": {"offset": 154200, "pending": ""}
}
```.

In [ ]:
import tempfile, os

# Simulación conceptual (sin depender del storage backend real del proyecto)
def simple_read_new_lines(file_path, last_offset):
    """Versión mínima de demostración: lee bytes nuevos desde el último offset."""
    with open(file_path, 'r', encoding='utf-8') as f:
        f.seek(last_offset)
        new_content = f.read()
        new_offset = f.tell()
    lines = new_content.splitlines()
    return lines, new_offset

with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as tmp:
    tmp.write("2026-01-15 10:00:00 Primer evento\n")
    tmp.write("2026-01-15 10:00:05 Segundo evento\n")
    tmp_path = tmp.name

offset = 0
lines, offset = simple_read_new_lines(tmp_path, offset)
print(f"Primera lectura -> lines={lines}, nuevo offset={offset}")

with open(tmp_path, 'a', encoding='utf-8') as f:
    f.write("2026-01-15 10:00:10 Tercer evento (nuevo)\n")

lines, offset = simple_read_new_lines(tmp_path, offset)
print(f"Segunda lectura (solo lo nuevo) -> lines={lines}, nuevo offset={offset}")

os.remove(tmp_path)

---
## 5. Registro de ficheros procesados (`parsing/registry.py`)

Complementario al offset: evita **reprocesar ficheros completos** que no han cambiado (útil para logs
rotados/cerrados, a diferencia del offset que es para ficheros activos que crecen).

In [ ]:

from parsing.registry import load_processed_files_registry, save_processed_files_registry

class FakeLocalStorage:
    """Storage mínimo en memoria para demostrar registry sin backend real."""
    def __init__(self):
        self._files = {}
    def exists(self, path):
        return path in self._files
    def read_json(self, path):
        import json
        return json.loads(self._files[path])
    def write_json(self, path, data):
        import json
        self._files[path] = json.dumps(data)

storage = FakeLocalStorage()
registry = load_processed_files_registry(storage, 'parsed_logs')
print(f"Registro inicial (vacío esperado): {registry}")

registry['logs/jde_2026_01_10.log'] = {
    'last_processed': '2026-01-15T10:30:00',
    'size_bytes': 154200,
    'records_count': 3421,
}
save_processed_files_registry(storage, 'parsed_logs', registry)

reloaded = load_processed_files_registry(storage, 'parsed_logs')
print(f"Registro tras guardar y recargar: {reloaded}")